## SparkSession Initialization

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName("SkyPath-ML")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")


In [3]:
df = spark.read.parquet("/data/processed/features/final")
print(f"Total rows: {df.count():,}, Columns: {len(df.columns)}")

from pyspark.sql.window import Window

window_spec = Window.partitionBy('Tail_Number', 'FlightDate').orderBy('CRSDepTime')

df = df.withColumn('cumulative_delay_prior',
    F.coalesce(
        F.sum('ArrDelay').over(
            window_spec.rowsBetween(Window.unboundedPreceding, -1)
        ),
        F.lit(0.0)
    ))


train_df = df.filter(F.col("Year") <= 2023)
test_df  = df.filter(F.col("Year") == 2024)

print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")

Total rows: 37,786,688, Columns: 57
Train: 30,821,441 | Test: 6,965,247


In [4]:
# Binary label: delay > 15 min
train_df = train_df.withColumn("label_cls", (F.col("ArrDelay") > 15).cast("int"))
test_df  = test_df.withColumn("label_cls",  (F.col("ArrDelay") > 15).cast("int"))

# Regression label: ArrDelay in minutes
train_df = train_df.withColumn("label_reg", F.col("ArrDelay"))
test_df  = test_df.withColumn("label_reg",  F.col("ArrDelay"))

# numeric features
numeric_features = [
    # time
    "Month", "DayOfWeek", "DepHour", "CRSDepTime",
    # flight data
    "Distance", "DepDelay",
    # Ripple
    "flight_leg", "prev_arr_delay", "cumulative_delay_prior",
    "inherited_delay",
    # weather
    "temperature", "dewpoint", "humidity",
    "wind_direction", "wind_speed", "visibility", "precipitation",
    
    "has_weather_data", "is_low_visibility", "is_high_wind", "has_precipitation",
    # historical statistics
    "route_total_flights", "route_avg_delay", "route_on_time_rate",
    "carrier_avg_delay", "carrier_on_time_rate",
    "origin_avg_dep_delay", "origin_std_dep_delay",
]

categorical_features = ["Reporting_Airline", "Origin", "Dest"]


In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, Imputer, VectorAssembler

# Impute null values
ripple_cols  = ["prev_arr_delay", "inherited_delay", "delay_recovery"]
weather_cols = ["temperature", "dewpoint", "humidity", "wind_direction",
                "wind_speed", "visibility", "precipitation"]
hist_cols    = ["route_avg_delay", "route_on_time_rate",
                "carrier_avg_delay", "carrier_on_time_rate",
                "origin_avg_dep_delay", "origin_std_dep_delay"]

imputer_zero = Imputer(
    inputCols=ripple_cols,
    outputCols=ripple_cols,
    strategy="mean"
)
imputer_mean = Imputer(
    inputCols=weather_cols + hist_cols,
    outputCols=weather_cols + hist_cols,
    strategy="mean"
)

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]
indexed_cat_cols = [f"{c}_idx" for c in categorical_features]

all_feature_cols = numeric_features + indexed_cat_cols
assembler = VectorAssembler(
    inputCols=all_feature_cols,
    outputCol="features",
    handleInvalid="skip" 
)

preprocess_pipeline = Pipeline(stages=[imputer_zero, imputer_mean] + indexers + [assembler])

preprocessor = preprocess_pipeline.fit(train_df)
train_features = preprocessor.transform(train_df)
test_features  = preprocessor.transform(test_df)

preprocessor.write().overwrite().save("/models/preprocessor")
print("Preprocessor saved.")


Preprocessor saved.


In [18]:
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

gbt_cls = GBTClassifier(
    featuresCol="features",
    labelCol="label_cls",
    maxIter=150,
    maxDepth=6,
    stepSize=0.05,
    maxBins=512,
    seed=42
)

print("Training classifier...")
cls_model = gbt_cls.fit(train_features)
cls_model.write().overwrite().save("/models/classifier")
print("Classifier saved.")

# Evaluate
cls_preds = cls_model.transform(test_features)

auc_eval = BinaryClassificationEvaluator(labelCol="label_cls", metricName="areaUnderROC")
f1_eval  = MulticlassClassificationEvaluator(labelCol="label_cls", metricName="f1")
acc_eval = MulticlassClassificationEvaluator(labelCol="label_cls", metricName="accuracy")

auc = auc_eval.evaluate(cls_preds)
f1  = f1_eval.evaluate(cls_preds)
acc = acc_eval.evaluate(cls_preds)

print(f"Classifier → AUC: {auc:.4f} | F1: {f1:.4f} | Accuracy: {acc:.4f}")


Training classifier...
Classifier saved.
Classifier → AUC: 0.9345 | F1: 0.8876 | Accuracy: 0.8910


In [20]:
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

gbt_reg = GBTRegressor(
    featuresCol="features",
    labelCol="label_reg",
    maxIter=50,
    maxDepth=6,
    stepSize=0.1,
    maxBins=512,
    seed=42
)

print("Training regressor...")
reg_model = gbt_reg.fit(train_features)
reg_model.write().overwrite().save("/models/regressor")
print("Regressor saved.")

# Evaluate
reg_preds = reg_model.transform(test_features)

rmse_eval = RegressionEvaluator(labelCol="label_reg", metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol="label_reg", metricName="mae")
r2_eval   = RegressionEvaluator(labelCol="label_reg", metricName="r2")

rmse = rmse_eval.evaluate(reg_preds)
mae  = mae_eval.evaluate(reg_preds)
r2   = r2_eval.evaluate(reg_preds)

print(f"Regressor → RMSE: {rmse:.2f} min | MAE: {mae:.2f} min | R²: {r2:.4f}")


Training regressor...
Regressor saved.
Regressor → RMSE: 20.70 min | MAE: 12.72 min | R²: 0.9199


In [21]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": all_feature_cols,
    "importance": cls_model.featureImportances.toArray()
}).sort_values("importance", ascending=False).head(15)

print("Top 15 features (classifier):")
print(feature_importance.to_string(index=False))

Top 15 features (classifier):
              feature  importance
             DepDelay    0.674789
             Dest_idx    0.077281
          temperature    0.056808
           Origin_idx    0.045234
Reporting_Airline_idx    0.023040
           wind_speed    0.018249
           visibility    0.017017
    has_precipitation    0.015535
        precipitation    0.012203
             dewpoint    0.010301
             humidity    0.009272
                Month    0.008813
             Distance    0.008414
       wind_direction    0.007524
           CRSDepTime    0.005687


In [9]:
# Pre-departure features (DepDelay removed — not yet known at booking time)
numeric_features_pre = [
    "Month", "DayOfWeek", "DepHour", "CRSDepTime",
    "Distance",
    "flight_leg", "prev_arr_delay", "inherited_delay", "cumulative_delay_prior",
    "temperature", "dewpoint", "humidity",
    "wind_direction", "wind_speed", "visibility", "precipitation",
    "has_weather_data", "is_low_visibility", "is_high_wind", "has_precipitation",
    "route_total_flights", "route_avg_delay", "route_on_time_rate",
    "carrier_avg_delay", "carrier_on_time_rate",
    "origin_avg_dep_delay", "origin_std_dep_delay",
]

assembler_pre = VectorAssembler(
    inputCols=numeric_features_pre + indexed_cat_cols,
    outputCol="features",
    handleInvalid="skip"
)

pre_pipeline = Pipeline(stages=[imputer_zero, imputer_mean] + indexers + [assembler_pre])
preprocessor_pre = pre_pipeline.fit(train_df)

train_features_pre = preprocessor_pre.transform(train_df)
test_features_pre  = preprocessor_pre.transform(test_df)

preprocessor_pre.write().overwrite().save("/models/pre_departure/preprocessor")
print("Pre-departure preprocessor saved.")

Pre-departure preprocessor saved.


In [23]:
gbt_cls_pre = GBTClassifier(
    featuresCol="features",
    labelCol="label_cls",
    maxIter=150,
    maxDepth=6,
    stepSize=0.05,
    maxBins=512,
    seed=42
)

print("Training pre-departure classifier...")
cls_model_pre = gbt_cls_pre.fit(train_features_pre)
cls_model_pre.write().overwrite().save("/models/pre_departure/classifier")

cls_preds_pre = cls_model_pre.transform(test_features_pre)
auc_pre = auc_eval.evaluate(cls_preds_pre)
f1_pre  = f1_eval.evaluate(cls_preds_pre)
acc_pre = acc_eval.evaluate(cls_preds_pre)

print(f"Pre-departure Classifier → AUC: {auc_pre:.4f} | F1: {f1_pre:.4f} | Accuracy: {acc_pre:.4f}")

Training pre-departure classifier...
Pre-departure Classifier → AUC: 0.8102 | F1: 0.7633 | Accuracy: 0.7805


In [24]:
gbt_reg_pre = GBTRegressor(
    featuresCol="features",
    labelCol="label_reg",
    maxIter=150,
    maxDepth=6,
    stepSize=0.05,
    maxBins=512,
    seed=42
)

print("Training pre-departure regressor...")
reg_model_pre = gbt_reg_pre.fit(train_features_pre)
reg_model_pre.write().overwrite().save("/models/pre_departure/regressor")

reg_preds_pre = reg_model_pre.transform(test_features_pre)
rmse_pre = rmse_eval.evaluate(reg_preds_pre)
mae_pre  = mae_eval.evaluate(reg_preds_pre)
r2_pre   = r2_eval.evaluate(reg_preds_pre)

print(f"Pre-departure Regressor → RMSE: {rmse_pre:.2f} min | MAE: {mae_pre:.2f} min | R²: {r2_pre:.4f}")

Training pre-departure regressor...
Pre-departure Regressor → RMSE: 64.62 min | MAE: 29.60 min | R²: 0.2192


In [25]:
all_feature_cols_pre = numeric_features_pre + indexed_cat_cols
fi_pre = pd.DataFrame({
    "feature":    all_feature_cols_pre,
    "importance": cls_model_pre.featureImportances.toArray()
}).sort_values("importance", ascending=False).head(15)
print("Top 15 features (pre-departure classifier):")
print(fi_pre.to_string(index=False))

Top 15 features (pre-departure classifier):
               feature  importance
              Dest_idx    0.183551
            Origin_idx    0.116980
        prev_arr_delay    0.113261
           temperature    0.082830
            CRSDepTime    0.078883
         precipitation    0.053680
                 Month    0.048944
 Reporting_Airline_idx    0.044620
            wind_speed    0.044432
cumulative_delay_prior    0.040067
            flight_leg    0.035737
              dewpoint    0.024859
       inherited_delay    0.023144
     has_precipitation    0.020012
            visibility    0.019579


## Load the existing models, run evaluation, and write the results to MongoDB

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.classification import GBTClassificationModel
from pyspark.ml.regression import GBTRegressionModel
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,
                                   MulticlassClassificationEvaluator,
                                   RegressionEvaluator)
import pandas as pd

spark = (SparkSession.builder
    .appName("SkyPath-Eval")
    .config("spark.driver.memory", "8g")
    .getOrCreate())

preprocessor     = PipelineModel.load("/models/preprocessor")
cls_model        = GBTClassificationModel.load("/models/classifier")
reg_model        = GBTRegressionModel.load("/models/regressor")

preprocessor_pre = PipelineModel.load("/models/pre_departure/preprocessor")
cls_model_pre    = GBTClassificationModel.load("/models/pre_departure/classifier")
reg_model_pre    = GBTRegressionModel.load("/models/pre_departure/regressor")

test_df = (spark.read.parquet("/data/processed/features/final")
           .filter(F.col("Year") == 2024))

from pyspark.sql.window import Window
window_spec = Window.partitionBy("Tail_Number", "FlightDate").orderBy("CRSDepTime")
test_df = test_df.withColumn("cumulative_delay_prior",
    F.coalesce(
        F.sum("ArrDelay").over(window_spec.rowsBetween(Window.unboundedPreceding, -1)),
        F.lit(0.0)
    ))

test_df = (test_df
    .withColumn("label_cls", (F.col("ArrDelay") > 15).cast("int"))
    .withColumn("label_reg",  F.col("ArrDelay")))

auc_eval  = BinaryClassificationEvaluator(labelCol="label_cls", metricName="areaUnderROC")
f1_eval   = MulticlassClassificationEvaluator(labelCol="label_cls", metricName="f1")
acc_eval  = MulticlassClassificationEvaluator(labelCol="label_cls", metricName="accuracy")
rmse_eval = RegressionEvaluator(labelCol="label_reg", metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol="label_reg", metricName="mae")
r2_eval   = RegressionEvaluator(labelCol="label_reg", metricName="r2")

test_post = preprocessor.transform(test_df)
auc  = auc_eval.evaluate(cls_model.transform(test_post))
f1   = f1_eval.evaluate(cls_model.transform(test_post))
acc  = acc_eval.evaluate(cls_model.transform(test_post))
rmse = rmse_eval.evaluate(reg_model.transform(test_post))
mae  = mae_eval.evaluate(reg_model.transform(test_post))
r2   = r2_eval.evaluate(reg_model.transform(test_post))

test_pre = preprocessor_pre.transform(test_df)
auc_pre  = auc_eval.evaluate(cls_model_pre.transform(test_pre))
f1_pre   = f1_eval.evaluate(cls_model_pre.transform(test_pre))
acc_pre  = acc_eval.evaluate(cls_model_pre.transform(test_pre))
rmse_pre = rmse_eval.evaluate(reg_model_pre.transform(test_pre))
mae_pre  = mae_eval.evaluate(reg_model_pre.transform(test_pre))
r2_pre   = r2_eval.evaluate(reg_model_pre.transform(test_pre))

all_feature_cols     = preprocessor.stages[-1].getInputCols()
all_feature_cols_pre = preprocessor_pre.stages[-1].getInputCols()
feature_importance   = pd.DataFrame({"feature": all_feature_cols,
                                     "importance": cls_model.featureImportances.toArray()}
                                    ).sort_values("importance", ascending=False).head(15)
fi_pre               = pd.DataFrame({"feature": all_feature_cols_pre,
                                     "importance": cls_model_pre.featureImportances.toArray()}
                                    ).sort_values("importance", ascending=False).head(15)

print(f"Post → AUC {auc:.4f} | F1 {f1:.4f} | RMSE {rmse:.2f}")
print(f"Pre  → AUC {auc_pre:.4f} | F1 {f1_pre:.4f} | RMSE {rmse_pre:.2f}")


Post → AUC 0.9345 | F1 0.8876 | RMSE 20.70
Pre  → AUC 0.8102 | F1 0.7633 | RMSE 64.62


In [17]:
import json
import datetime
from pymongo import MongoClient

eval_post = {
    "model_type":  "post_departure",
    "description": "with DepDelay feature",
    "train_years": "2019-2023",
    "test_year":   "2024",
    "features":    all_feature_cols,
    "classifier": {
        "auc":      round(auc,  4),
        "f1":       round(f1,   4),
        "accuracy": round(acc,  4),
    },
    "regressor": {
        "rmse": round(rmse, 2),
        "mae":  round(mae,  2),
        "r2":   round(r2,   4),
    },
    "top_features": feature_importance["feature"].tolist(),
}

eval_pre = {
    "model_type":  "pre_departure",
    "description": "without DepDelay feature",
    "train_years": "2019-2023",
    "test_year":   "2024",
    "features":    all_feature_cols_pre,
    "classifier": {
        "auc":      round(auc_pre,  4),
        "f1":       round(f1_pre,   4),
        "accuracy": round(acc_pre,  4),
    },
    "regressor": {
        "rmse": round(rmse_pre, 2),
        "mae":  round(mae_pre,  2),
        "r2":   round(r2_pre,   4),
    },
    "top_features": fi_pre["feature"].tolist(),
}

combined = {
    "generated_at":  datetime.datetime.utcnow().isoformat(),
    "post_departure": eval_post,
    "pre_departure":  eval_pre,
}
with open("/results/model_evaluation.json", "w") as f:
    json.dump(combined, f, indent=2, ensure_ascii=False)
print("model_evaluation.json saved.")


client = MongoClient("mongodb://skypath-mongo:27017/")
db = client["skypath"]
now = datetime.datetime.utcnow()

db.model_metrics.replace_one(
    {"model_type": "post_departure"},
    {**eval_post, "timestamp": now},
    upsert=True,
)
db.model_metrics.replace_one(
    {"model_type": "pre_departure"},
    {**eval_pre, "timestamp": now},
    upsert=True,
)
print("model_metrics upserted (2 documents).")


model_evaluation.json saved.
model_metrics upserted (2 documents).
  + overview.json
  + carriers.json
  + airports.json
  + temporal.json
  + attribution.json
  + ripple.json
  + routes.json

Done: 7 imported, 0 skipped.


In [ ]:
analysis_names = ["overview", "carriers", "airports", "temporal",
                  "attribution", "ripple", "routes"]
imported, skipped = 0, 0
for name in analysis_names:
    path = f"/results/analysis/{name}.json"
    try:
        with open(path) as f:
            data = json.load(f)
        db.analysis.replace_one(
            {"name": name},
            {"name": name, "data": data, "timestamp": now},
            upsert=True,
        )
        print(f"  + {name}.json")
        imported += 1
    except FileNotFoundError:
        print(f"  - {name}.json not found, skipping")
        skipped += 1

print(f"\nDone: {imported} imported, {skipped} skipped.")